<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/ndvi_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --pre xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url = 'https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
border = border = (
    ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")
    .filterBounds(roi)
)


In [ ]:
map.addLayer(border)

In [ ]:
def ndvi(img):
  prob = img.select('probability')
  clear_pixels = prob.lt(20)
  sr = img.select('B.*').multiply(0.0001)
  index = sr.normalizedDifference(['B8','B4']).rename('ndvi')
  bands = sr.select(['B4','B8'],['red','nir'])
  stack = ee.Image.cat([bands, index]).updateMask(clear_pixels)
  return stack

In [ ]:
sen2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .linkCollection(
        ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY"), 'probability'
    )
    .filterDate('2025','2026')
    .filterBounds(roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .map(ndvi)
    .median()
)

sen2

In [ ]:
from shapely.geometry import shape

In [ ]:
vec = shape(roi.getInfo())
vec

In [ ]:
vec_gdf = geemap.ee_to_gdf(border)
vec = vec_gdf.geometry.iloc[0]
vec

In [ ]:
from xee import helpers

In [ ]:
from jinja2.nodes import ExprStmt

In [ ]:
import geopandas as gpd

In [ ]:
grid_params = helpers.fit_geometry(
    geometry = vec,
    grid_crs = 'EPSG:3857',
    grid_scale = (33, 4)
)

In [ ]:
import xarray as xr

In [ ]:
ds = xr.open_dataset(
    sen2,
    engine = 'ee',
    **grid_params

)

In [ ]:
ds

In [ ]:
ds = ds.squeeze('time').drop_vars('time') * 1

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
ds

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (14,4))
ds.red.plot(x = 'x', y = 'y', ax = ax[0], robust = True)
ds.nir.plot(x = 'x', y = 'y', ax = ax[1], robust = True)
ds.ndvi.plot(x = 'x', y = 'y', ax = ax[2], cmap = 'RdYlGn')

plt.tight_layout()
plt.show()

In [ ]:
plt.tight_layout()

red_vals = ds.red.values.flatten()
nir_vals = ds.nir.values.flatten()
ndvi_vals = ds.ndvi.values.flatten()

In [ ]:
fig, ax = plt.subplots(figsize = (6,4))

scatter = ax.scatter(x = red_vals, y = nir_vals, c = ndvi_vals, cmap = 'RdYlGn')

cbar = fig.colorbar(scatter, ax = ax, cmap = 'RdYlGn', pad = 0.02)

cbar.set_label('NDVI', fontsize = 12)

cbar.ax.tick_params(labelsize = 12)
ax.set_ylabel('NIR Reflectance', fontsize = 12)
ax.set_xlabel('Red Reflectance', fontsize = 12)
ax.set_title('NIR-RED Scatter Plot', fontsize = 15)
ax.tick_params(axis = 'both', direction = 'in', right = True, top = True, labelsize = 12)
ax.grid(ls = ':', alpha = 0.7)